# Dev Notebook for Text-Only Modeling

In [27]:
import random
import torch
import io
import pyarrow as pa
import os
import copy
from sacred import Experiment
from PIL import Image
from tqdm import tqdm
import numpy as np
import skimage.io as skio
import matplotlib.pyplot as plt
from refer import REFER

from meter.modules.heads import Pooler

from torch.utils.data import DataLoader
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule

from torch.optim import AdamW

from transformers import ElectraTokenizer
from transformers import AutoModel, AutoModelForSequenceClassification

from refcoco_utils import get_bounded_subimage
from refcoco_utils import _config
from refcoco_utils import _loss_names

from meter.transforms import keys_to_transforms
from meter.config import ex
from meter.modules import METERTransformerSS
from meter.datamodules.multitask_datamodule import MTDataModule
from meter.datasets.base_dataset import BaseDataset

In [2]:
refer_root = "/home/claytonfields/nlp/code/data/coco"

In [5]:
_config = {  
    "exp_name":"finetune_mrpc",
    "seed" : 0,
    # "datasets" : ["coco", "vg", "sbu", "gcc"],
    # "datasets" : ["coco", "vg"],
    "datasets" : ["coco"],
    "loss_names" :{'itm': 0,
    'mlm': 0,
    'mpp': 0,
    'vqa': 0,
    'vcr': 0,
    'vcr_qar': 0,
    'nlvr2': 0,
    'irtr': 0,
    'contras': 0,
    'snli': 0,
    'ref': 0,
    'mrpc':1
    },
    "batch_size" : 32,  # this is a desired batch size; pl trainer will accumulate gradients when per step batch is smaller.

    # Image setting
    "image_encoder" : "facebook/deit-tiny-patch16-224",
    "random_init_vision_encoder" : False,
    "image_encoder_hidden_size" : 192,
    "image_size" : 224,
    "patch_size" : 16,
    "draw_false_image" : 1,
    "image_only" : False,
    "resolution_before" : 224,
    "train_transform_keys" : ["imagenet"],
    "val_transform_keys" : ["imagenet"],

    # Text Setting
    "text_encoder" : "google/electra-small-discriminator",
    "random_init_text_encoder" : False,
    "text_encoder_hidden_size" : 256,
    "vocab_size" : 30522,
    "whole_word_masking" : False, # note that whole_word_masking does not work for RoBERTa
    "mlm_prob" : 0.15,
    "draw_false_text" : 0,
    "vqav2_label_size" : 3129,
    "max_text_len" : 128,

    # CrossLayer Setting
    "num_cross_layers" : 6,
    "cross_layer_hidden_size" : 256,
    "num_cross_layer_heads" : 4,
    "cross_layer_mlp_ratio" : 4,
    "cross_layer_drop_rate" : 0.1,
    
    # Optimizer Setting
    "optim_type" : "adamw",
    "learning_rate" : 1e-5,
    "weight_decay" : 0.01,
    "decay_power" : 1,
    "max_epoch" : 3,
    "max_steps" : 100000,
    "warmup_steps" : 10000,
    "end_lr" : 0,
    "lr_mult_head" : 5,  # multiply lr for downstream heads
    "lr_mult_cross_modal" : 5,  # multiply lr for the cross-modal module

    # Encoder Settings
    "freeze_image_encoder" : True,
    "freeze_text_encoder" : False,
    'freeze_cross_modal_layers' : True,
    

    # Downstream Setting
    "get_recall_metric" : False,
    
    'freeze' : True,
    
    "model_type" : "METER",

    # PL Trainer Setting
    "resume_from" : None,
    "fast_dev_run" : False,
    "val_check_interval" : 1.0,
    "test_only" : False,

    "data_root" : "/home/claytonfields/nlp/code/meter/data/arrow",
    "log_dir" : "result",
    "per_gpu_batchsize" : 32,  # you should define this manually with per_gpu_batch_size:#
    "num_gpus" : 1,
    "num_nodes" : 1,
    # "load_path" : "/home/claytonfields/nlp/code/meter/result/mlm_itm_seed0_from_/meter_electra_small_deit_tiny_p16_is224_bs288_is1M/checkpoints/epoch=43-step=898039.ckpt",
    "load_path" : '/home/claytonfields/nlp/code/meter/result/mlm_itm_deit_fr_electra_fr_is224_ps16_bs336_pgbs84_ts100k/checkpoints/epoch=5-step=96215.ckpt',
    "num_workers" : 12,
    "precision" : 32
}

In [7]:
model = METERTransformerSS(_config)
# dm = MTDataModule(_config, dist=False)
# dm.prepare_data()
# dm.setup('train')

Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Text Encoder

### Classification Head

In [8]:
from transformers import AutoTokenizer
from datasets import load_dataset
from torchtext.datasets import mrpc

In [9]:
data_dict = load_dataset('glue', 'mrpc', split='train').to_dict()

In [10]:
data_dict.keys()

dict_keys(['sentence1', 'sentence2', 'label', 'idx'])

In [11]:
data_dict['idx'].__len__()#[-1]

3668

In [37]:
class GlueDataset(torch.utils.data.Dataset):
    def __init__(self, task, split, tokenizer):

        self.task = task
        self.split = split
        self.tokenizer = tokenizer

        self.data_dict = load_dataset('glue', self.task, split=self.split).to_dict()
        self.sentence1 = self.data_dict['sentence1']
        self.sentence2 = self.data_dict['sentence2']
        self.label = self.data_dict['label']
        self.idx = self.data_dict['idx']

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, index):

        sent1 = self.sentence1[index]
        sent2 = self.sentence2[index]
        label = self.label[index]
        # idx = self.idx[index]

        ret = tokenizer(
            sent1, 
            sent2,
            max_length=60,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        ret = {k: v.squeeze() for k,v in ret.items()}
        ret['labels'] = label
        return ret

    

In [13]:
tokenizer = AutoTokenizer.from_pretrained(model.text_encoder.config._name_or_path)
tokenizer

ElectraTokenizerFast(name_or_path='google/electra-small-discriminator', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [38]:
ds = GlueDataset('mrpc', 'train', tokenizer)
ds

In [39]:
dl = DataLoader(ds, batch_size=10)
dl

In [16]:
text_pooler = Pooler(_config['text_encoder_hidden_size'])
loss_fn = F.cross_entropy

In [17]:
# Objective
batch = next(iter(dl))
labels = labels = batch.pop('label',None)
hidden_state = model.text_encoder(**batch).last_hidden_state#.squeeze()
cls_feat = text_pooler(hidden_state)
# cls_feat.shape
logits = model.mrpc_classifier(cls_feat)
loss = loss_fn(logits, labels)
loss

tensor(0.7157, grad_fn=<NllLossBackward0>)

In [18]:
class GlueDataModule(LightningDataModule):
    def __init__(self, config, task, batch_size):
        super().__init__()

        self.task = task
        self.batch_size = batch_size
        self.tokenizer = AutoTokenizer.from_pretrained(config['text_encoder'])
        
    def set_train_dataset(self):
        self.train_dataset = load_dataset('glue', self.task, split='train')

    def set_val_dataset(self):
        self.val_dataset =  load_dataset('glue', self.task, split='val')

    def set_test_dataset(self):
         self.text_dataset = load_dataset('glue', self.task, split='test')
        
    def setup(self, stage: str):
        self.set_train_dataset()
        self.set_val_dataset()
        self.set_test_dataset()

    def train_dataloader(self):
        loader = DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            # num_workers=self.num_workers,
            # pin_memory=True,
            # collate_fn=self.collate_fn,
        )
        return loader

    def val_dataloader(self):
        loader = DataLoader(
            self.val_dataset,
            batch_size=self.eval_batch_size,
            shuffle=False,
            # num_workers=self.num_workers,
            # pin_memory=True,
            # collate_fn=self.collate_fn,
        )
        return loader
        
    def test_dataloader(self):
        loader = DataLoader(
            self.test_dataset,
            batch_size=self.eval_batch_size,
            shuffle=False,
            # num_workers=self.num_workers,
            # pin_memory=True,
            # collate_fn=self.collate_fn,
        )
        return loader

In [19]:
config = copy.deepcopy(_config)
pl.seed_everything(_config["seed"])
model = METERTransformerSS(config)
model.current_tasks = ['mrpc']


dm = GlueDataModule(_config, 'mrpc', 32)

pl.seed_everything(_config["seed"])

exp_name = f'{_config["exp_name"]}'

os.makedirs(_config["log_dir"], exist_ok=True)
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    save_top_k=1,
    verbose=True,
    monitor="val/the_metric",
    mode="max",
    save_last=True,
)
logger = pl.loggers.TensorBoardLogger(
    _config["log_dir"],
    name=f'{exp_name}_seed{_config["seed"]}_from_{_config["load_path"].split("/")[-1][:-5]}',
)

lr_callback = pl.callbacks.LearningRateMonitor(logging_interval="step")
callbacks = [checkpoint_callback, lr_callback]

num_gpus = (
    _config["num_gpus"]
    if isinstance(_config["num_gpus"], int)
    else len(_config["num_gpus"])
)

grad_steps = max(_config["batch_size"] // (
    _config["per_gpu_batchsize"] * num_gpus * _config["num_nodes"]
), 1)

max_steps = _config["max_steps"] if _config["max_steps"] is not None else None

trainer = pl.Trainer(
    devices=num_gpus,
    num_nodes=_config["num_nodes"],
    precision=_config["precision"],
    # accelerator="ddp",
    benchmark=True,
    deterministic=True,
    max_epochs=_config["max_epoch"] if max_steps is None else 1000,
    max_steps=max_steps,
    callbacks=callbacks,
    logger=logger,
    #prepare_data_per_node=False,
    #replace_sampler_ddp=False,
    accumulate_grad_batches=grad_steps,
    log_every_n_steps=10,
    # flush_logs_every_n_steps=10,
#     resume_from_checkpoint=_config["resume_from"],
    # weights_summary="top",
    fast_dev_run=_config["fast_dev_run"],
    val_check_interval=_config["val_check_interval"],
)

# log_dir = logger.log_dir
# eval_file = 'eval.txt'
# eval_path = os.path.join(log_dir, eval_file )
# setattr(model, f"eval_path", eval_path)
# f = open(eval_path,'w') 
# f.close()

if not _config["test_only"]:
    trainer.fit(model, datamodule=dm)
else:
    trainer.test(model, datamodule=dm)


Seed set to 0
Some weights of ViTModel were not initialized from the model checkpoint at facebook/deit-tiny-patch16-224 and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Seed set to 0
/home/claytonfields/anaconda3/envs/meter-test/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:668: You passed `deterministic=True` and `benchmark=True`. Note that PyTorch ignores torch.backends.cudnn.deterministic=True when torch.backends.cudnn.benchmark=True.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
You are using a CUDA device ('NVIDIA GeForce RTX 3080 Laptop GPU') that has Tensor Cores. To properly utilize th

ValueError: Unknown split "val". Should be one of ['train', 'validation', 'test'].

In [20]:
text_encoder = model.text_encoder
text_encoder

ElectraModel(
  (embeddings): ElectraEmbeddings(
    (word_embeddings): Embedding(30522, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (embeddings_project): Linear(in_features=128, out_features=256, bias=True)
  (encoder): ElectraEncoder(
    (layer): ModuleList(
      (0-11): 12 x ElectraLayer(
        (attention): ElectraAttention(
          (self): ElectraSelfAttention(
            (query): Linear(in_features=256, out_features=256, bias=True)
            (key): Linear(in_features=256, out_features=256, bias=True)
            (value): Linear(in_features=256, out_features=256, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): ElectraSelfOutput(
            (dense): Linear(in_features=256, out_features=256, bias=True)
            (LayerNorm): LayerNorm((

In [23]:
text_encoder.save_pretrained('temp') 

In [28]:
text_encoder = AutoModelForSequenceClassification.from_pretrained('temp')

Some weights of ElectraForSequenceClassification were not initialized from the model checkpoint at temp and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
batch = next(iter(dl))

In [41]:
batch

{'input_ids': tensor([[  101,  2572,  3217,  5831,  5496,  2010,  2567,  1010,  3183,  2002,
           2170,  1000,  1996,  7409,  1000,  1010,  1997,  9969,  4487, 23809,
           3436,  2010,  3350,  1012,   102,  7727,  2000,  2032,  2004,  2069,
           1000,  1996,  7409,  1000,  1010,  2572,  3217,  5831,  5496,  2010,
           2567,  1997,  9969,  4487, 23809,  3436,  2010,  3350,  1012,   102,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
         [  101,  9805,  3540, 11514,  2050,  3079, 11282,  2243,  1005,  1055,
           2077,  4855,  1996,  4677,  2000,  3647,  4576,  1999,  2687,  2005,
           1002,  1016,  1012,  1019,  4551,  1012,   102,  9805,  3540, 11514,
           2050,  4149, 11282,  2243,  1005,  1055,  1999,  2786,  2005,  1002,
           6353,  2509,  2454,  1998,  2853,  2009,  2000,  3647,  4576,  2005,
           1002,  1015,  1012,  1022,  4551,  1999,  2687,  1012,   102,     0],
         [  101,  2027,  

In [42]:
text_encoder(**batch)

SequenceClassifierOutput(loss=tensor(0.6922, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0488,  0.0591],
        [-0.0371,  0.0814],
        [-0.0517,  0.0860],
        [-0.0520,  0.0791],
        [-0.0666,  0.0767],
        [-0.0401,  0.0930],
        [-0.0494,  0.0902],
        [-0.0661,  0.0982],
        [-0.0466,  0.0893],
        [-0.0434,  0.0553]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

## Import Calssification Head?

In [ ]:
model_type = model.text_encoder.base_model_prefix
model_name = model_type.capitalize()
model_type = 'bert'
model_name = model_type.capitalize()

In [ ]:
exec_string = f'from transformers.models.{model_type}.modeling_{model_type} import {model_name}ClassificationHead'

In [ ]:
from transformers.models.electra.modeling_electra import ElectraClassificationHead

In [ ]:
exec(exec_string)

## Image Encoder

In [ ]:
model.image_encoder(